# 0. Imports

In [182]:
 %pip install "scipy<1.11"

Note: you may need to restart the kernel to use updated packages.


In [183]:
%pip install pandas numpy scikit-learn lightgbm matplotlib seaborn statsmodels

Note: you may need to restart the kernel to use updated packages.


In [184]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
import lightgbm
import scipy
import statsmodels
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# 1. Questions


# 2. Preprocessing 

In [185]:
test = pd.read_json('../data/test.json')[['bathrooms', 'bedrooms', 'price', 'features']]
train = pd.read_json('../data/train.json')[['bathrooms', 'bedrooms', 'price', 'features']]
test.shape, train.shape

((74659, 4), (49352, 4))

# 3. Data analysis part 2

In [186]:
test.columns, train.columns

(Index(['bathrooms', 'bedrooms', 'price', 'features'], dtype='object'),
 Index(['bathrooms', 'bedrooms', 'price', 'features'], dtype='object'))

In [187]:
test.head()

,bathrooms,bedrooms,price,features
0,1.0,1,2950,"[Elevator, Laundry in Building, Laundry in Uni..."
1,1.0,2,2850,"[Pre-War, Dogs Allowed, Cats Allowed]"
2,1.0,0,2295,"[Pre-War, Dogs Allowed, Cats Allowed]"
3,1.0,2,2900,"[Hardwood Floors, Dogs Allowed, Cats Allowed]"
5,1.0,1,3254,"[Roof Deck, Doorman, Elevator, Fitness Center,..."


In [188]:
train.head()

,bathrooms,bedrooms,price,features
4,1.0,1,2400,"[Dining Room, Pre-War, Laundry in Building, Di..."
6,1.0,2,3800,"[Doorman, Elevator, Laundry in Building, Dishw..."
9,1.0,2,3495,"[Doorman, Elevator, Laundry in Building, Laund..."
10,1.5,3,3000,[]
15,1.0,0,2795,"[Doorman, Elevator, Fitness Center, Laundry in..."


In [189]:
train['features'].head(10)

4     [Dining Room, Pre-War, Laundry in Building, Di...
6     [Doorman, Elevator, Laundry in Building, Dishw...
9     [Doorman, Elevator, Laundry in Building, Laund...
10                                                   []
15    [Doorman, Elevator, Fitness Center, Laundry in...
16    [Doorman, Elevator, Loft, Dishwasher, Hardwood...
18    [Fireplace, Laundry in Unit, Dishwasher, Hardw...
19    [Elevator, Laundry in Building, Dishwasher, Ha...
23                                    [Hardwood Floors]
32                         [Cats Allowed, Dogs Allowed]
Name: features, dtype: object

In [190]:
def clear_features(df_iterator):
    all_features = []

    for _, row in df_iterator:
        features_str = str(row["features"])
        features_str = re.sub(r'[\[\]\'\"\s]', '', features_str)

        features = [f.strip() for f in features_str.split(',') if f.strip()]
        all_features.extend(features)
    return all_features

In [191]:
all_features = clear_features(train.iterrows())

In [192]:
unique_features = set(all_features)
len(unique_features)

1546

In [193]:
feature_counter = Counter(all_features)
top_20_features = feature_counter.most_common(20)
top_20_features = [f[0] for f in top_20_features]
top_20_features

['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

In [194]:
for feature in top_20_features:
    train[feature] = train["features"].str.contains(feature, regex=False).astype(int)
    test[feature] = test["features"].str.contains(feature, regex=False).astype(int)

In [195]:
features_list = top_20_features + ['bathrooms', 'bedrooms']
len(features_list)

22

In [196]:
new_train = train.drop(columns='features')
new_test = test.drop(columns='features')

lower = new_train['price'].quantile(0.01)
upper = new_train['price'].quantile(0.99)
new_test = new_test[new_test['price'].between(lower, upper) & (new_test['bedrooms'] < 5) & (new_test['bathrooms'] < 5)]
new_train = new_train[new_train['price'].between(lower, upper) & (new_train['bedrooms'] < 5) & (new_train['bathrooms'] < 5)]

# 4. Models implementation — Linear regression

надо добавить параметры solver в котором будут значения: "analitical", "sgd", "batch", "minibatch"
добавить l1_ratio

## 4.1 класс линейной регрессии 
с реализованным SGD и аналитическим обучением

регулициями (задание 5)

batch-GD и minibatch-GD (задание 11) 

In [197]:
class customLinReg:
    def __init__(self, solver="sgd", lr = 0.01, epochs = 100, random_state = None, reg=None, l1_ratio = 0.5, batch_size = 32):
        self.solver = solver
        self.lr = lr
        self.epochs = epochs
        self.random_state = random_state
        self.reg = reg
        self.l1_ratio = l1_ratio
        self.batch_size = batch_size

    def __init_weights(self, features_count):
        if self.random_state != None:
            self.random_generator = np.random.RandomState(self.random_state)
        else:
            self.random_generator = np.random
        self.weights = self.random_generator.normal(loc=0.0, scale=0.01, size=1 + features_count)
    
    def __pred(self, xi):
        xi = np.array(xi, dtype=float)
        return np.dot(xi, self.weights[1:]) + self.weights[0]
    
    def __ridge(self):
        return (1 - self.l1_ratio) * 2 * self.weights[1:]
    def __lasso(self):
        return self.l1_ratio * np.sign(self.weights[1:])
    def __elastic(self):
        return self.__ridge() + self.__lasso()
    def __reg_term(self):
        if self.reg == None: return 0
        elif self.reg == 'ridge': return self.__ridge()
        elif self.reg == 'lasso': return self.__lasso()
        elif self.reg == 'elastic': return self.__elastic()
        
    def predict(self, X):
        predicts = np.dot(X, self.weights[1:]) + self.weights[0]
        return predicts

    def __analytical(self, X_train, y_train):

        X_b = np.hstack([np.ones((X_train.shape[0], 1)), X_train])

        XTX = np.dot(X_b.T, X_b)

        if self.reg is not None and self.reg == 'ridge':
            if self.reg == 'ridge':
                n_features = X_b.shape[1]
                I = np.eye(n_features)
                I[0, 0] = 0

                XTX = XTX + (1 - self.l1_ratio) * I
        self.weights = np.dot(np.dot(np.linalg.pinv(XTX), X_b.T), y_train)

        return self
    
    def __sgd(self, X_train, y_train):
        n_samples, n_features = X_train.shape
        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)
            for i in indices:
                xi = X_train[i]
                target = y_train[i]
                pred = self.__pred(xi)

                dw = (pred - target) * xi + self.__reg_term()
                db = (pred - target)

                self.weights[1:] -= self.lr * dw
                self.weights[0] -= self.lr * db

    def __batch(self, X_train, y_train):  
        n_samples, n_features = X_train.shape
        for _ in range(self.epochs):
            reg_term = self.__reg_term()
            pred = self.predict(X_train)
            dw = (1/n_samples) * np.dot(X_train.T, (pred - y_train)) + (1/n_samples) * reg_term
            db = (1/n_samples) * np.sum(pred - y_train)

            self.weights[0] -= self.lr * db
            self.weights[1:] -= self.lr * dw

    def __mini_batch(self, X_train, y_train):
        n_samples, n_features = X_train.shape

        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)

            for start in range(0, n_samples, self.batch_size):
                batch_idx = indices[start:start + self.batch_size]

                X_batch = X_train[batch_idx]
                y_batch = y_train[batch_idx]

                pred = self.predict(X_batch)

                dw = (1 / len(X_batch)) * np.dot(X_batch.T, (pred - y_batch)) + (1/self.batch_size) * self.__reg_term()
                db = (1 / len(X_batch)) * np.sum(pred - y_batch)

                self.weights[1:] -= self.lr * dw
                self.weights[0] -= self.lr * db


    def fit(self, X_train, y_train):

        X_train = np.array(X_train, dtype=float)
        y_train = np.array(y_train, dtype=float)
        self.__init_weights(X_train.shape[1])

        if self.solver == "sgd":
            self.__sgd(X_train, y_train) 
        elif self.solver == "analytical":
            self.__analytical(X_train, y_train)
        elif self.solver == "batch":
            self.__batch(X_train, y_train)
        elif self.solver == "minibatch": 
            self.__mini_batch(X_train, y_train)


## 4.2 Что такое детерменированная модель?
это модель которая при одних весах и входных данных всегда выдает один и тот же результат (то есть отсутсвует элементы случайности)

чтобы модель сделать детерменнированной нужно добавить в конструктор класса random_state, который устанавливается как сид для генератора 

## 4.3 функция для подсчета R2

In [198]:
def my_R2(y_true, y_pred):
    mean_true = np.mean(y_true)
    mean_mse = np.sum((y_true - mean_true)**2)
    pred_mse = np.sum((y_true - y_pred)**2)
    return 1 - (pred_mse/mean_mse)

## 4.4 функция для предсказаний и создания таблиц с метриками моделей

In [199]:
# def get_model_table(models_dict, X_tr, X_tst, y_tr, y_tst):
#     results = []
#     for name, model in models_dict.items():
#         model.fit(X_tr, y_tr)
#         pred_train = model.predict(X_tr)
#         pred_test = model.predict(X_tst)
        
#         results.append({
#             ('Model', ''): name,
#             ('MAE', 'Train'): mean_absolute_error(y_tr, pred_train),
#             ('MAE', 'Test'): mean_absolute_error(y_tst, pred_test),
#             ('RMSE', 'Train'): root_mean_squared_error(y_tr, pred_train),
#             ('RMSE', 'Test'): root_mean_squared_error(y_tst, pred_test),
#             ('R2', 'Train'): my_R2(y_tr, pred_train),
#             ('R2', 'Test'): my_R2(y_tst, pred_test)
#         })
    
#     df = pd.DataFrame(results)
#     df.columns = pd.MultiIndex.from_tuples(df.columns)
#     return df.set_index(('Model', ''))


def get_model_table(models_dict, X_tr, X_tst, y_tr, y_tst):                
    results = []                                                           
    model_names = []                                                       
    for name, model in models_dict.items():                               
        model.fit(X_tr, y_tr)                                              
        pred_train = model.predict(X_tr)                                   
        pred_test = model.predict(X_tst)                                   
                                                                            
        model_names.append(name)                                           
        results.append({                                                   
             ('MAE', 'Train'): mean_absolute_error(y_tr, pred_train),      
             ('MAE', 'Test'): mean_absolute_error(y_tst, pred_test),       
             ('RMSE', 'Train'): mean_squared_error(y_tr, pred_train, squared=False), 
            ('RMSE', 'Test'): mean_squared_error(y_tst, pred_test, squared=False),   
            ('R2', 'Train'): my_R2(y_tr, pred_train),                      
            ('R2', 'Test'): my_R2(y_tst, pred_test)                        
        })                                                                 
                                                                           
    df = pd.DataFrame(results, index=model_names)                          
    df.columns = pd.MultiIndex.from_tuples(df.columns)                     
    df.index.name = 'Model'                                                
    return df                                                             


In [200]:
X_train = new_train.drop(columns='price')
y_train = new_train['price']


In [201]:
X_test = new_test.drop(columns='price')
y_test = new_test['price']

In [202]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(solver='svd'),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (analytical)': customLinReg(solver='analytical'),
    'Custom LinReg (SGD)': customLinReg(epochs=20),
    "Custom Ridge(analytical)": customLinReg(solver="analytical", reg='ridge'),
    'Custom Ridge': customLinReg(reg='ridge'),
    'Custom Lasso': customLinReg(reg='lasso'),
    'Custom ElasticNet': customLinReg(reg='elastic'),
}

## таблица с метриками

In [203]:
linregs_table = get_model_table(
    {
        'Linear Regression': LinearRegression(),
        'Custom LinReg (analytical)': customLinReg(solver='analytical'),
        'Custom LinReg (SGD)': customLinReg(epochs=20),
    },
    X_train,
    X_test,
    y_train,
    y_test
)
linregs_table 

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           731.168838  734.139387  1054.031454  1054.055141   
Custom LinReg (analytical)  731.168838  734.139387  1054.031454  1054.055141   
Custom LinReg (SGD)         897.769745  898.638688  1174.803093  1174.200164   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.544573  0.546077  
Custom LinReg (analytical)  0.544573  0.546077  
Custom LinReg (SGD)         0.434227  0.436699

# 5. Regularized models implementation — Ridge, Lasso, ElasticNet

обучение моделей и сохранение метрик

In [204]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(solver='svd'),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (analytical)': customLinReg(solver='analytical'),
    'Custom LinReg (SGD)': customLinReg(epochs=20),
    "Custom Ridge(analytical)": customLinReg(solver="analytical", reg='ridge'),
    'Custom Ridge': customLinReg(reg='ridge'),
    'Custom Lasso': customLinReg(reg='lasso'),
    'Custom ElasticNet': customLinReg(reg='elastic'),
}

In [205]:
regulized_table = get_model_table(
    models,
    X_train, 
    X_test,
    y_train,
    y_test
)
regulized_table

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           731.168838  734.139387  1054.031454  1054.055141   
Ridge                       731.166952  734.137740  1054.031458  1054.055634   
Lasso                       730.933107  734.040323  1054.075982  1054.115856   
ElasticNet                  813.501222  817.812533  1196.125743  1199.002590   
Custom LinReg (analytical)  731.168838  734.139387  1054.031454  1054.055141   
Custom LinReg (SGD)         748.282174  751.301451  1145.089707  1145.069377   
Custom Ridge(analytical)    731.167895  734.138562  1054.031455  1054.055386   
Custom Ridge                879.065409  883.107221  1233.175784  1236.333426   
Custom Lasso                730.237608  732.673293  1075.569648  1075.105632   
Custom ElasticNet           864.496039  868.507847  1315.884970  1318.885613   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.544573  0.546077  
Ridge                       0.544573  0.546076  
Lasso                       0.544534  0.546024  
ElasticNet                  0.413503  0.412651  
Custom LinReg (analytical)  0.544573  0.546077  
Custom LinReg (SGD)         0.462485  0.464303  
Custom Ridge(analytical)    0.544573  0.546076  
Custom Ridge                0.376607  0.375508  
Custom Lasso                0.525770  0.527765  
Custom ElasticNet           0.290181  0.289326

# 6. Feature normalization

## 6.1 примеры где нужна нормализация
- 

## 6.2 формулы minmax и standartization

## 6.3 Классы my_MinMaxScaler и my_StandartScaler

In [206]:
class my_MinMaxScaler:
    def __init__(self):
        self.min_ = None
        self.max_ = None

    def fit(self, X):
        X = np.array(X)
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self
    
    def transform(self, X):
        X = np.asarray(X)
        range_ = self.max_ - self.min_
        range_ = np.where(range_ == 0, 1, range_)
        X_scaled = (X - self.min_) / (range_)
        # X_scaled[np.isnan(X_scaled)] = 0  
        return X_scaled

    
    def fit_transform(self, X):
        return self.fit(X).transform(X)


In [207]:
class my_StandartScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
    
    def fit(self, X):
        X = np.array(X)

        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self
    
    def transform(self, X):
        X = np.asarray(X)
        self.std_ = np.where(self.std_ == 0, 1, self.std_)

        X_scaled = (X - self.mean_) / self.std_
        return X_scaled
       
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


In [208]:
mm_scaler = my_MinMaxScaler()
std_scaler = my_StandartScaler()

## 6.4 инициализация MinMaxScaler и StandartScaler из sklearn

In [209]:
sk_mm_scaler = MinMaxScaler()
sk_std_scaler = StandardScaler()

нормализация данных

In [210]:
X_train_MM = mm_scaler.fit_transform(X_train)
X_test_MM = mm_scaler.transform(X_test)

In [211]:
X_train_MMsk = sk_mm_scaler.fit_transform(X_train)
X_test_MMsk = sk_mm_scaler.transform(X_test)

In [212]:
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

In [213]:
X_train_stdsk = sk_std_scaler.fit_transform(X_train)
X_test_stdsk = sk_std_scaler.transform(X_test)

## 6.5 сравнение результатов


In [214]:
np.allclose(X_train_std, X_train_stdsk)

True

In [215]:
np.allclose(X_train_MM, X_train_MMsk)

True

# 7. Fit custom and sklearn models with normalized data

In [216]:
minmax_table = get_model_table(models,
        X_train_MM,
        X_test_MM,
        y_train,
        y_test
        )
minmax_table

MAE                      RMSE  \
                                  Train         Test        Train   
Model                                                               
Linear Regression            731.168838   734.139387  1054.031454   
Ridge                        731.163099   734.134567  1054.032986   
Lasso                        730.980507   734.098808  1054.121803   
ElasticNet                  1014.980488  1017.609480  1436.673277   
Custom LinReg (analytical)   731.168838   734.139387  1054.031454   
Custom LinReg (SGD)          738.309669   741.685772  1056.899535   
Custom Ridge(analytical)     731.165819   734.136883  1054.031839   
Custom Ridge                1051.280907  1053.964175  1478.512138   
Custom Lasso                 746.954516   749.584416  1057.952779   
Custom ElasticNet           1025.845061  1028.458028  1496.154500   

                                               R2            
                                   Test     Train      Test  
Model                                                        
Linear Regression           1054.055141  0.544573  0.546077  
Ridge                       1054.068011  0.544571  0.546065  
Lasso                       1054.210832  0.544495  0.545942  
ElasticNet                  1439.032332  0.153888  0.153948  
Custom LinReg (analytical)  1054.055141  0.544573  0.546077  
Custom LinReg (SGD)         1056.760748  0.542091  0.543743  
Custom Ridge(analytical)    1054.061209  0.544572  0.546071  
Custom Ridge                1480.942533  0.103889  0.103950  
Custom Lasso                1057.824525  0.541178  0.542824  
Custom ElasticNet           1498.490012  0.082376  0.082590

In [217]:
standart_table = get_model_table(models,
        X_train_std,
        X_test_std,
        y_train,
        y_test
        )
standart_table

MAE                     RMSE               \
                                 Train        Test        Train         Test   
Model                                                                          
Linear Regression           731.168838  734.139387  1054.031454  1054.055141   
Ridge                       731.168636  734.139248  1054.031455  1054.055206   
Lasso                       731.098901  734.112653  1054.035105  1054.065814   
ElasticNet                  756.158059  759.680735  1092.316048  1093.629115   
Custom LinReg (analytical)  731.168838  734.139387  1054.031454  1054.055141   
Custom LinReg (SGD)         773.975764  775.867217  1086.875974  1087.199607   
Custom Ridge(analytical)    731.168736  734.139316  1054.031455  1054.055174   
Custom Ridge                768.139064  772.034446  1093.990038  1093.953463   
Custom Lasso                781.804372  784.838711  1119.584659  1120.898618   
Custom ElasticNet           823.077805  826.660254  1177.425065  1179.864268   

                                  R2            
                               Train      Test  
Model                                           
Linear Regression           0.544573  0.546077  
Ridge                       0.544573  0.546076  
Lasso                       0.544570  0.546067  
ElasticNet                  0.510888  0.511352  
Custom LinReg (analytical)  0.544573  0.546077  
Custom LinReg (SGD)         0.515748  0.517081  
Custom Ridge(analytical)    0.544573  0.546077  
Custom Ridge                0.509387  0.511062  
Custom Lasso                0.486163  0.486679  
Custom ElasticNet           0.431699  0.431252

# 8. Overfit models

## 8.2 создание полиномиальных признаков

In [218]:
pf = PolynomialFeatures(10)
X_train_poly = pf.fit_transform(X_train.iloc[:, :2])
X_test_poly = pf.transform(X_test.iloc[:, :2])

In [219]:
X_train_poly = np.hstack((X_train_poly, X_train.iloc[:, 2:]))
X_test_poly = np.hstack((X_test_poly, X_test.iloc[:, 2:]))

In [220]:
poly_table = get_model_table(
    models,
    X_train_poly, 
    X_test_poly,
    y_train,
    y_test
)

/opt/homebrew/Caskroom/miniconda/base/envs/sklearntest/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 24972795345.914146, tolerance: 11739750.338619603
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Caskroom/miniconda/base/envs/sklearntest/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:530: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 27279902992.131035, tolerance: 11739750.338619603
  model = cd_fast.enet_coordinate_descent(
/var/folders/rd/09hc85s918sgj8b6x7wprt4r0000gn/T/ipykernel_46722/2296027794.py:64: RuntimeWarning: overflow encountered in multiply
  dw = (pred - target) * xi + self.__reg_term()
/var/folders/rd/09hc85s918sgj8b6x7wprt4r0000gn/T/ipykernel_46722/2296027794.py:64: RuntimeWarning: invalid value encountered in multiply
  dw 

ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [ ]:
poly_table

MAE                       RMSE                \
                        Train          Test        Train          Test   
Model                                                                    
Linear Regression  698.831835  8.483630e+07  1007.196983  9.350214e+09   
Ridge              699.274631  7.338086e+02  1007.058716  3.565222e+03   

                         R2                
                      Train          Test  
Model                                      
Linear Regression  0.577397 -3.621060e+13  
Ridge              0.577513 -4.264604e+00

In [ ]:
mylinreg = customLinReg(solver='analytical')
mylinreg.fit(X_train_poly, y_train)

# 9. Native models

# 10. Compare results

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Custom LinReg (SGD)': customLinReg(epochs=20),
    'Custom Ridge': customLinReg(reg='ridge'),
    'Custom Lasso': customLinReg(reg='lasso'),
    'Custom ElasticNet': customLinReg(reg='elastic'),
}

results_df = get_model_table(models, X_train, X_test, y_train, y_test)
results_df

MAE                      RMSE               \
                           Train         Test        Train         Test   
Model                                                                     
Linear Regression     729.137477   732.845517  1049.893799  1052.051328   
Ridge                 729.135472   732.843732  1049.893803  1052.051469   
Lasso                 728.896775   732.742846  1049.938632  1052.104741   
ElasticNet            811.829824   816.794522  1191.214767  1195.370206   
Custom LinReg (SGD)   738.669340   742.666935  1055.969917  1058.699667   
Custom Ridge         1147.845643  1152.785669  1391.066735  1396.504022   
Custom Lasso          715.738384   719.621163  1060.798475  1062.921054   
Custom ElasticNet     878.933442   883.984066  1287.145312  1291.503286   

                           R2            
                        Train      Test  
Model                                    
Linear Regression    0.540808  0.541577  
Ridge                0.540808  0.541577  
Lasso                0.540769  0.541530  
ElasticNet           0.408869  0.408169  
Custom LinReg (SGD)  0.535478  0.535765  
Custom Ridge         0.193881  0.192250  
Custom Lasso         0.531220  0.532055  
Custom ElasticNet    0.309826  0.309150

In [ ]:
models_std = {
    'Linear Regression + StdScaler': LinearRegression(),
    'Ridge + StdScaler': Ridge(),
    'Lasso + StdScaler': Lasso(),
    'ElasticNet + StdScaler': ElasticNet(),
}

results_std_df = get_model_table(models_std, X_train_stdsk, X_test_stdsk, y_train, y_test)
results_std_df

MAE                     RMSE  \
                                    Train        Test        Train   
Model                                                                
Linear Regression + StdScaler  729.137477  732.845517  1049.893799   
Ridge + StdScaler              729.137273  732.845370  1049.893799   
Lasso + StdScaler              729.066603  732.818085  1049.897459   
ElasticNet + StdScaler         753.628701  757.827145  1087.393422   

                                                  R2            
                                      Test     Train      Test  
Model                                                           
Linear Regression + StdScaler  1052.051328  0.540808  0.541577  
Ridge + StdScaler              1052.051338  0.540808  0.541577  
Lasso + StdScaler              1052.059161  0.540805  0.541570  
ElasticNet + StdScaler         1090.098323  0.507420  0.507820

# 11. Addition task